# 07 — Iterate

Data-driven component replication.

**What you learn:**
- `iterate='^path'`: replicate a component once per child in data
- `^.?attr`: resolve an attribute from the current data node
- Data is a `Bag` of nodes with attributes

**Prerequisites:** 05_components, 04_pointers

In [ ]:
from IPython.display import HTML

from genro_bag.resolvers import FileResolver
from genro_builders.builder import component
from genro_builders.contrib.html import HtmlBuilder
from genro_builders.manager import ReactiveManager
from genro_toolbox import set_sync

set_sync()

In [ ]:
class CatalogBuilder(HtmlBuilder):
    @component(main_tag='div', sub_tags='')
    def product_card(self, comp, **kwargs):
        card = comp.div(_class='card')
        card.h3('^.?name')
        card.p('^.?price', _class='price')
        card.p('^.?description')
        card.p('^.?category', _class='category')

In [ ]:
class ProductCatalog(ReactiveManager):
    def on_init(self):
        self.page = self.register_builder('page', CatalogBuilder)

    def main(self, source):
        # FileResolver loads style.css lazily at render time (pull model)
        source.head().style(FileResolver('style.css'))
        body = source.body()
        body.h1('Product Catalog')
        body.p('Each card generated by iterate.')
        catalog = body.div(_class='catalog')
        catalog.product_card(iterate='^products')

In [ ]:
app = ProductCatalog()
# FileResolver with as_bag=True: Bag node format preserved for iterate
app.local_store('page').set_resolver(
    'products', FileResolver('products.json', as_bag=True),
)
app.run()
HTML(app.page.render())